# A/B Test Significance Analysis
#1. Introduction
**Мета:** порахувати статистичну значущість чотирьох ключових конверсійних
метрик A/B-тесту через Python (замість онлайн-калькуляторів) і підготувати
CSV-файл для візуалізації в Tableau.

**Метрики, які аналізуємо:**
- add_payment_info / session
- add_shipping_info / session
- begin_checkout / session
- new_accounts / session

**Джерело даних:** той самий набір даних з BigQuery, що і в завданні
*Create Your A/B Testing Tool* — тягнемо його напряму запитом, без
ручного експорту в CSV.

**План ноутбука:**
1. Introduction
2. Environment Setup
3. Підключення до BigQuery і завантаження даних
4. Визначення метрик
5. Z-тест для двох пропорцій
6. Функція розрахунку значущості для довільного зрізу
7. Розрахунок в тоталі по кожному тесту
8. Розрахунок в розрізах
9. Збереження результату в CSV
10. Висновки та посилання

#2. Environment Setup

In [36]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
from google.cloud import bigquery
from google.colab import auth

#3. Підключення до BigQuery і завантаження даних

In [37]:
auth.authenticate_user()

PROJECT_ID = 'data-analytics-mate'
client = bigquery.Client(project=PROJECT_ID)

query = """
WITH
  session_info AS (
    SELECT
      s.date,
      s.ga_session_id,
      sp.country,
      sp.device,
      sp.continent,
      sp.channel,
      ab.test,
      ab.test_group
    FROM `DA.ab_test` ab
    JOIN `DA.session` s
      ON ab.ga_session_id = s.ga_session_id
    JOIN `DA.session_params` sp
      ON ab.ga_session_id = sp.ga_session_id
  ),




session_with_orders AS (
SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT o.ga_session_id) AS sessions_with_orders
FROM `DA.order` o
JOIN session_info
  ON o.ga_session_id = session_info.ga_session_id
  GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
),


events AS (
SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      ep.event_name,
      COUNT(DISTINCT ep.ga_session_id) AS event_cnt  -- сесії з подією, а не кількість подій
FROM `DA.event_params` ep
JOIN session_info
ON ep.ga_session_id = session_info.ga_session_id
GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      ep.event_name
),


session AS (
  SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT session_info.ga_session_id) AS session_cnt
  FROM session_info
  GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
),


account AS (
  SELECT
  session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT acs.ga_session_id) AS new_account_cnt
  FROM `data-analytics-mate.DA.account_session` acs
  JOIN session_info
  ON session_info.ga_session_id = acs.ga_session_id
  GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
      )




SELECT
      session_with_orders.date,
      session_with_orders.country,
      session_with_orders.device,
      session_with_orders.continent,
      session_with_orders.channel,
      session_with_orders.test,
      session_with_orders.test_group,
      'session_with_orders' AS event_name,
      session_with_orders.sessions_with_orders AS value
FROM session_with_orders


UNION ALL


SELECT
      events.date,
      events.country,
      events.device,
      events.continent,
      events.channel,
      events.test,
      events.test_group,
      event_name,
      event_cnt AS value
FROM events


UNION ALL


SELECT
      session.date,
      session.country,
      session.device,
      session.continent,
      session.channel,
      session.test,
      session.test_group,
      'session' AS event_name,
      session_cnt AS value
FROM session


UNION ALL


SELECT
      account.date,
      account.country,
      account.device,
      account.continent,
      account.channel,
      account.test,
      account.test_group,
      'new_account' AS event_name,
      new_account_cnt AS value
FROM account
"""

df = client.query(query).to_dataframe(create_bqstorage_client=False)
df.head(11)

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-01,Ecuador,desktop,Americas,Organic Search,2,2,session_with_orders,1
1,2020-11-01,Uruguay,mobile,Americas,Organic Search,2,2,session_with_orders,1
2,2020-11-01,Kenya,desktop,Africa,Organic Search,2,1,session_with_orders,1
3,2020-11-01,Serbia,desktop,Europe,Paid Search,2,1,session_with_orders,1
4,2020-11-01,Malta,mobile,Europe,Social Search,2,2,session_with_orders,1
5,2020-11-01,Puerto Rico,desktop,Americas,Organic Search,2,2,session_with_orders,1
6,2020-11-01,Jamaica,desktop,Americas,Organic Search,2,1,session_with_orders,1
7,2020-11-02,Jordan,desktop,Asia,Organic Search,2,1,session_with_orders,1
8,2020-11-02,Bahrain,desktop,Asia,Undefined,2,2,session_with_orders,1
9,2020-11-02,North Macedonia,desktop,Europe,Direct,2,1,session_with_orders,1


In [38]:
df[df['event_name'] == 'add_payment_info'].groupby(['test', 'test_group'])['value'].sum()

test  test_group
1     1              964
      2              983
2     1             1096
      2             1125
3     1             1771
      2             1788
4     1             1850
      2             1841
Name: value, dtype: Int64

In [39]:
sessions_by_group = (
    df[df['event_name'] == 'session']
    .groupby(['test', 'test_group'])['value']
    .sum()
    .unstack('test_group')
    .fillna(0)  # Заповнюємо пропуски нулями, якщо якоїсь групи немає
)

srm_rows = []
for test_number, row in sessions_by_group.iterrows():
    # Приводим до списку чисельних значень
    observed = row.astype(float).values

    # Виконуємо тест Хі-квадрат
    chi2, p_srm = stats.chisquare(f_obs=observed)

    srm_rows.append({
        'test': test_number,
        'sessions_group_1': int(observed[0]),
        'sessions_group_2': int(observed[1]),
        'share_group_1': observed[0] / observed.sum() if observed.sum() > 0 else 0,
        'p_value_srm': p_srm,
        'srm_detected': p_srm < 0.01,
    })

srm_df = pd.DataFrame(srm_rows)
srm_df

,test,sessions_group_1,sessions_group_2,share_group_1,p_value_srm,srm_detected
0,1,45362,45193,0.500933,0.574386,False
1,2,50637,50244,0.501948,0.215962,False
2,3,70047,70439,0.498605,0.295630,False
3,4,105079,105141,0.499853,0.892435,False


# Перевірка самої формули на числах із прикладу-файлу до завдання (test 2, add_payment_info).
# Важливо: приклад рахує події без DISTINCT, тому з нашими виправленими даними він уже не збігається.
# Тут ми перевіряємо лише, що функція дає той самий z і p на тих самих вхідних числах.

In [40]:
# Швидка перевірка структури даних перед розрахунками
print('Колонки:', list(df.columns))
print('Унікальні тести:', sorted(df['test'].unique()))
print('Унікальні групи:', sorted(df['test_group'].unique()))
print('Приклади event_name:', df['event_name'].unique()[:20])

Колонки: ['date', 'country', 'device', 'continent', 'channel', 'test', 'test_group', 'event_name', 'value']
Унікальні тести: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Унікальні групи: [np.int64(1), np.int64(2)]
Приклади event_name: ['session_with_orders' 'new_account' 'session' 'add_to_cart' 'page_view'
 'view_search_results' 'session_start' 'user_engagement' 'view_promotion'
 'begin_checkout' 'scroll' 'view_item' 'add_shipping_info' 'first_visit'
 'add_payment_info' 'select_item' 'select_promotion' 'click'
 'view_item_list']


#4. Визначення метрик

Уся подальша логіка працюватиме циклом по цьому списку: якщо додати сюди п'яту метрику (наприклад, add_to_cart / session), решту коду міняти не потрібно.

In [41]:
METRICS = [
    {'name': 'add_payment_info / session',  'numerator_event': 'add_payment_info',  'denominator_event': 'session'},
    {'name': 'add_shipping_info / session', 'numerator_event': 'add_shipping_info', 'denominator_event': 'session'},
    {'name': 'begin_checkout / session',    'numerator_event': 'begin_checkout',    'denominator_event': 'session'},
    {'name': 'new_accounts / session',      'numerator_event': 'new_account',       'denominator_event': 'session'},
]

In [42]:
# Швидка перевірка щоб переконатися, що всі event_name з METRICS і session дійсно є в даних (інакше метрика випаде в подальших розрахунках)
needed_events = {'session'} | {m['numerator_event'] for m in METRICS}
present_events = set(df['event_name'].unique())

print('Відсутні в даних:', needed_events - present_events)

Відсутні в даних: set()


#5. Z-тест для двох пропорцій

Для кожної метрики порівнюємо частку конверсій у групі 1 і групі 2 через z-тест для двох незалежних пропорцій

p1 = x1/n1, p2 = x2/n2 — конверсія в кожній групі
p_pool = (x1+x2)/(n1+n2) — об'єднана (pooled) конверсія
SE = sqrt(p_pool*(1-p_pool)*(1/n1+1/n2)) — стандартна похибка
z = (p1-p2)/SE
p_value — двостороннє значення з нормального розподілу
Якщо p_value < 0.05 (рівень значущості α=5%) → різниця між групами статистично значуща.

In [43]:
def two_proportion_ztest(x1, n1, x2, n2):
    """
    Z-тест для двох пропорцій.
    x1, x2 — numerator (кількість "успіхів") у групі 1 і 2
    n1, n2 — denominator (розмір вибірки) у групі 1 і 2
    Повертає (z_stat, p_value)
    """
    if n1 == 0 or n2 == 0:
        return np.nan, np.nan

    p1, p2 = x1 / n1, x2 / n2
    p_pool = (x1 + x2) / (n1 + n2)
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))

    if se == 0:
        return np.nan, np.nan

    z = (p1 - p2) / se
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    return z, p_value


# Перевірка формули на числах із прикладу-файлу до завдання (test 2, add_payment_info):
z_check, p_check = two_proportion_ztest(2409, 50244, 2344, 50637)
print(f'z = {z_check:.6f}, p = {p_check:.6f}')  # очікується z ≈ 1.240994, p ≈ 0.214608 (НЕ значуще)

z = 1.240994, p = 0.214608


### Перевірка самої формули на числах із прикладу-файлу до завдання (test 2, add_payment_info).
### Важливо: приклад рахує події без DISTINCT, тому з нашими виправленими даними він уже не збігається.
### Тут ми перевіряємо лише, що функція дає той самий z і p на тих самих вхідних числах.

In [44]:
def diff_confidence_interval(x1, n1, x2, n2, confidence=0.95):
    """
    Довірчий інтервал для різниці пропорцій p2 - p1 (unpooled SE).
    Повертає (diff, ci_low, ci_high) у частках, не у відсотках.
    """
    p1, p2 = x1 / n1, x2 / n2
    z_crit = stats.norm.ppf(1 - (1 - confidence) / 2)
    se_diff = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
    diff = p2 - p1
    return diff, diff - z_crit * se_diff, diff + z_crit * se_diff


diff, low, high = diff_confidence_interval(2409, 50244, 2344, 50637)
print(f'diff = {diff*100:.3f} п.п., 95% CI [{low*100:.3f}; {high*100:.3f}] п.п.')

diff = -0.166 п.п., 95% CI [-0.427; 0.096] п.п.


#6. Функція розрахунку значущості для довільного зрізу даних

Функція приймає будь-яку підмножину датафрейму (весь `df` — для тоталу, або `df`, відфільтрований по країні/пристрою — для розрізу) і рахує всі 4 метрики для кожного тесту в цьому підмножині. Кількість метрик і кількість тестів **ніде не захардкоджені** — все йде циклами по унікальних значеннях і по списку `METRICS`.

In [45]:
def calculate_significance(df_subset, metrics, cut_type='total', cut_value='total'):
    results = []

    # 1. Сумуємо value по test / test_group / event_name
    agg = (
        df_subset
        .groupby(['test', 'test_group', 'event_name'], as_index=False)['value']
        .sum()
    )

    # 2. "Розвертаємо" event_name у колонки, щоб зручно брати numerator/denominator
    pivot = agg.pivot_table(
        index=['test', 'test_group'],
        columns='event_name',
        values='value',
        fill_value=0
    ).reset_index()

    for test_number in sorted(pivot['test'].unique()):
        test_slice = pivot[pivot['test'] == test_number].sort_values('test_group')
        groups = test_slice['test_group'].tolist()

        if len(groups) < 2:
            continue  # немає з чим порівнювати (лише 1 група в цьому зрізі)

        group_a, group_b = groups[0], groups[1]
        row_a = test_slice[test_slice['test_group'] == group_a].iloc[0]
        row_b = test_slice[test_slice['test_group'] == group_b].iloc[0]

        # Цикл по METRICS — саме тут немає хардкоду кількості метрик
        for metric in metrics:
            num_event = metric['numerator_event']
            den_event = metric['denominator_event']

            if num_event not in pivot.columns or den_event not in pivot.columns:
                continue

            n1, d1 = row_a[num_event], row_a[den_event]
            n2, d2 = row_b[num_event], row_b[den_event]

            if d1 == 0 or d2 == 0:
                continue

            rate_1 = n1 / d1
            rate_2 = n2 / d2
            z_stat, p_value = two_proportion_ztest(n1, d1, n2, d2)
            diff, ci_low, ci_high = diff_confidence_interval(n1, d1, n2, d2)
            metric_change_pct = (rate_2 - rate_1) / rate_1 * 100 if rate_1 else np.nan

            results.append({
                'test_number': test_number,
                'metric': metric['name'],
                'numerator_event': num_event,
                'denominator_event': den_event,
                'cut_type': cut_type,
                'cut_value': cut_value,
                'group_1': group_a,
                'group_2': group_b,
                'numerator_count_1': n1,
                'denominator_count_1': d1,
                'conversion_rate_1': rate_1,
                'numerator_count_2': n2,
                'denominator_count_2': d2,
                'conversion_rate_2': rate_2,
                'metric_change_pct': metric_change_pct,
                'diff_pp': diff * 100,          # різниця між групами, у відсоткових пунктах
                'ci_low_pp': ci_low * 100,      # нижня межа 95% інтервалу
                'ci_high_pp': ci_high * 100,    # верхня межа 95% інтервалу
                'z_stat': z_stat,
                'p_value': p_value,
                'significant': bool(p_value < 0.05) if pd.notna(p_value) else None
            })

    return results

- agg - звичайний groupby().sum(), підсумовує value по всіх датах/країнах/пристроях/каналах, які не входять у зріз (для тоталу, по всіх одразу).
- pivot_table перетворює довгий формат (event_name як значення) на широкий, тепер кожна подія стає окремою колонкою, і можна просто взяти row['add_payment_info'] і row['session'].
- Внутрішній цикл for metric in metrics - саме він робить кількість метрик динамічною.
- cut_type/cut_value це поля, які потім у Tableau дозволять фільтрувати "тотал" окремо від розрізів по країні/пристрою.

#7. Розрахунок в тоталі по кожному тесту

In [46]:
all_results = []

# Базовий розрахунок в тоталі, без розрізів
all_results += calculate_significance(df, METRICS, cut_type='total', cut_value='total')

print(f'Отримано {len(all_results)} рядків (тести × метрики)')

Отримано 16 рядків (тести × метрики)


In [47]:
pd.DataFrame(all_results)

,test_number,metric,numerator_event,denominator_event,cut_type,cut_value,group_1,group_2,numerator_count_1,denominator_count_1,...,numerator_count_2,denominator_count_2,conversion_rate_2,metric_change_pct,diff_pp,ci_low_pp,ci_high_pp,z_stat,p_value,significant
0,1,add_payment_info / session,add_payment_info,session,total,total,1,2,964.0,45362.0,...,983.0,45193.0,0.021751,2.352276,0.049989,-0.138957,0.238935,-0.518551,0.604074,False
1,1,add_shipping_info / session,add_shipping_info,session,total,total,1,2,1860.0,45362.0,...,1992.0,45193.0,0.044078,7.497264,0.307414,0.044517,0.570311,-2.291930,0.021910,True
2,1,begin_checkout / session,begin_checkout,session,total,total,1,2,1862.0,45362.0,...,1992.0,45193.0,0.044078,7.381800,0.303005,0.040043,0.565967,-2.258498,0.023915,True
3,1,new_accounts / session,new_account,session,total,total,1,2,3823.0,45362.0,...,3681.0,45193.0,0.081451,-3.354299,-0.282692,-0.641788,0.076404,1.542883,0.122859,False
4,2,add_payment_info / session,add_payment_info,session,total,total,1,2,1096.0,50637.0,...,1125.0,50244.0,0.022391,3.448865,0.074648,-0.106460,0.255757,-0.807895,0.419151,False
5,2,add_shipping_info / session,add_shipping_info,session,total,total,1,2,2194.0,50637.0,...,2136.0,50244.0,0.042513,-1.882068,-0.081546,-0.331681,0.168588,0.638943,0.522860,False
6,2,begin_checkout / session,begin_checkout,session,total,total,1,2,2195.0,50637.0,...,2137.0,50244.0,0.042532,-1.880854,-0.081531,-0.331720,0.168659,0.638682,0.523030,False
7,2,new_accounts / session,new_account,session,total,total,1,2,4165.0,50637.0,...,4184.0,50244.0,0.083274,1.241934,0.102152,-0.237896,0.442199,-0.588793,0.556000,False
8,3,add_payment_info / session,add_payment_info,session,total,total,1,2,1771.0,70047.0,...,1788.0,70439.0,0.025384,0.398058,0.010064,-0.154273,0.174401,-0.120029,0.904461,False
9,3,add_shipping_info / session,add_shipping_info,session,total,total,1,2,2915.0,70047.0,...,2889.0,70439.0,0.041014,-1.443484,-0.060070,-0.268211,0.148070,0.565667,0.571620,False


**Поправка на множинні порівняння (метод Холма).** У кожному тесті ми перевіряємо одразу 4 метрики. Якщо кожну перевіряти на рівні 5%, шанс отримати хоча б один випадковий «значущий» результат зростає приблизно до 19%. Поправка Холма коригує p-value так, щоб цей шанс залишався 5% на весь тест. Остаточну значущість визначаємо за p_value_adj < 0.05.

In [48]:
totals_df = pd.DataFrame(all_results)

totals_df['p_value_adj'] = (
    totals_df.groupby('test_number')['p_value']
    .transform(lambda p: multipletests(p, method='holm')[1])
)
totals_df['significant_adj'] = totals_df['p_value_adj'] < 0.05

totals_df[['test_number', 'metric', 'diff_pp', 'ci_low_pp', 'ci_high_pp',
           'p_value', 'p_value_adj', 'significant', 'significant_adj']]

,test_number,metric,diff_pp,ci_low_pp,ci_high_pp,p_value,p_value_adj,significant,significant_adj
0,1,add_payment_info / session,0.049989,-0.138957,0.238935,0.604074,0.604074,False,False
1,1,add_shipping_info / session,0.307414,0.044517,0.570311,0.021910,0.087639,True,False
2,1,begin_checkout / session,0.303005,0.040043,0.565967,0.023915,0.087639,True,False
3,1,new_accounts / session,-0.282692,-0.641788,0.076404,0.122859,0.245719,False,False
4,2,add_payment_info / session,0.074648,-0.106460,0.255757,0.419151,1.000000,False,False
5,2,add_shipping_info / session,-0.081546,-0.331681,0.168588,0.522860,1.000000,False,False
6,2,begin_checkout / session,-0.081531,-0.331720,0.168659,0.523030,1.000000,False,False
7,2,new_accounts / session,0.102152,-0.237896,0.442199,0.556000,1.000000,False,False
8,3,add_payment_info / session,0.010064,-0.154273,0.174401,0.904461,1.000000,False,False
9,3,add_shipping_info / session,-0.060070,-0.268211,0.148070,0.571620,1.000000,False,False


#8. Розрахунок в розрізах (country / device / continent / channel)

Щоб зробити і показати результат детальніше, рахуємо ту саму значущість окремо для кожного значення в country, device, continent, channel. Список розрізів є масивом: додати новий розріз (наприклад date) можна одним рядком, не змінюючи функцію calculate_significance.

In [49]:
CUT_DIMENSIONS = ['country', 'device', 'continent', 'channel']

for dim in CUT_DIMENSIONS:
    for value in df[dim].dropna().unique():
        subset = df[df[dim] == value]
        all_results += calculate_significance(subset, METRICS, cut_type=dim, cut_value=value)

print(f'Всього рядків після розрізів: {len(all_results)}')

Всього рядків після розрізів: 1956


**Що тут відбувається:** функція calculate_significance з кроку 6 просто викликається ще раз для кожного унікального значення в кожному з 4 вимірів, але вже на відфільтрованому підмножині df. Наприклад, для country == 'Ukraine' вона порахує ті ж 4 метрики × (скільки там тестів трапилось в Україні), з cut_type='country', cut_value='Ukraine'.
Важливий нюанс: якщо в якомусь зрізі (наприклад, рідкісна країна) є дані лише по одній групі тесту, рядок просто не додасться (в функції стоїть if len(groups) < 2: continue), це нормально й очікувано.

In [50]:
results_df = pd.DataFrame(all_results)

# Скільки рядків не порахувалось коректно (через NaN у p_value)?
print('Рядків з NaN p_value:', results_df['p_value'].isna().sum())

# Прибираємо їх з фінального файлу — вони не несуть статистичної інформації
results_df = results_df.dropna(subset=['p_value'])
print('Залишилось рядків:', len(results_df))

Рядків з NaN p_value: 145
Залишилось рядків: 1811


**Звідки NaN.** 145 порівнянь (усі в розрізі country) отримали NaN, бо в обох групах не було жодної сесії з подією: конверсія 0% проти 0%, похибка дорівнює нулю, і z-тест порахувати неможливо. Такі рядки не несуть інформації, тому їх прибрано. У першій версії проєкту частина NaN мала ще одну причину: через COUNT без DISTINCT «конверсія» в дрібних країнах могла перевищувати 100%. Після виправлення SQL це неможливо.

**Розрізи - це дослідницький аналіз.** Тут сотні порівнянь без поправки на множинні порівняння, тому частина «значущих» результатів випадкова. Висновки робимо лише з тоталів, а значущі розрізи сприймаємо як гіпотези для наступних тестів.

In [51]:
# подивись звідки рядки відсіялись
nan_rows = pd.DataFrame(all_results)
nan_rows = nan_rows[nan_rows['p_value'].isna()]
nan_rows['cut_type'].value_counts()

,count
cut_type,
country,145


In [52]:
nan_rows[['numerator_count_1', 'numerator_count_2']].sum(axis=1).max()

0.0

#9. Формування та збереження фінального CSV

In [53]:
# 1. Додаємо скориговану значущість з тоталів (розділ 7)
results_df = results_df.merge(
    totals_df[['test_number', 'metric', 'cut_type', 'p_value_adj', 'significant_adj']],
    on=['test_number', 'metric', 'cut_type'],
    how='left'
)

# 2. Готова мітка для кольору в Tableau
def result_label(row):
    if row['cut_type'] == 'total':
        return 'Significant' if row['significant_adj'] == True else 'Not significant'
    return 'Significant (exploratory)' if row['significant'] else 'Not significant'

results_df['result'] = results_df.apply(result_label, axis=1)
print(results_df.groupby('cut_type')['result'].value_counts())


results_df = results_df.sort_values(['cut_type', 'test_number', 'metric']).reset_index(drop=True)

results_df.to_csv('ab_test_significance_results.csv', index=False)
results_df.head(20)

cut_type   result                   
channel    Not significant                75
           Significant (exploratory)       5
continent  Not significant                94
           Significant (exploratory)       2
country    Not significant              1520
           Significant (exploratory)      51
device     Not significant                44
           Significant (exploratory)       4
total      Not significant                16
Name: count, dtype: int64


,test_number,metric,numerator_event,denominator_event,cut_type,cut_value,group_1,group_2,numerator_count_1,denominator_count_1,...,metric_change_pct,diff_pp,ci_low_pp,ci_high_pp,z_stat,p_value,significant,p_value_adj,significant_adj,result
0,1,add_payment_info / session,add_payment_info,session,channel,Organic Search,1,2,291.0,15675.0,...,-13.847516,-0.257073,-0.545741,0.031594,1.745184,0.080953,False,NaN,NaN,Not significant
1,1,add_payment_info / session,add_payment_info,session,channel,Paid Search,1,2,225.0,11777.0,...,4.321970,0.082571,-0.270271,0.435413,-0.458639,0.646493,False,NaN,NaN,Not significant
2,1,add_payment_info / session,add_payment_info,session,channel,Social Search,1,2,117.0,3883.0,...,-7.043356,-0.212226,-0.955868,0.531417,0.559539,0.575794,False,NaN,NaN,Not significant
3,1,add_payment_info / session,add_payment_info,session,channel,Undefined,1,2,126.0,3336.0,...,23.145072,0.874184,-0.084985,1.833354,-1.784279,0.074378,False,NaN,NaN,Not significant
4,1,add_payment_info / session,add_payment_info,session,channel,Direct,1,2,205.0,10691.0,...,14.761877,0.283059,-0.100830,0.666948,-1.446625,0.148002,False,NaN,NaN,Not significant
5,1,add_shipping_info / session,add_shipping_info,session,channel,Organic Search,1,2,629.0,15675.0,...,3.310663,0.132849,-0.305389,0.571087,-0.594160,0.552405,False,NaN,NaN,Not significant
6,1,add_shipping_info / session,add_shipping_info,session,channel,Paid Search,1,2,485.0,11777.0,...,2.125430,0.087529,-0.421883,0.596941,-0.336760,0.736298,False,NaN,NaN,Not significant
7,1,add_shipping_info / session,add_shipping_info,session,channel,Social Search,1,2,174.0,3883.0,...,14.311548,0.641311,-0.304492,1.587113,-1.327964,0.184190,False,NaN,NaN,Not significant
8,1,add_shipping_info / session,add_shipping_info,session,channel,Undefined,1,2,153.0,3336.0,...,31.580922,1.448406,0.378289,2.518523,-2.648340,0.008089,True,NaN,NaN,Significant (exploratory)
9,1,add_shipping_info / session,add_shipping_info,session,channel,Direct,1,2,419.0,10691.0,...,7.864055,0.308207,-0.226034,0.842448,-1.131323,0.257919,False,NaN,NaN,Not significant


In [54]:
# Додатковий файл: компактне порівняння 4 метрик по кожному тесту
# (лише тотал, без розрізів, для швидкого візуального порівняння)
summary_df = results_df[results_df['cut_type'] == 'total'].copy()

summary_df = summary_df[[
    'test_number', 'metric',
    'conversion_rate_1', 'conversion_rate_2', 'metric_change_pct',
    'diff_pp', 'ci_low_pp', 'ci_high_pp',
    'z_stat', 'p_value', 'p_value_adj', 'significant', 'significant_adj', 'result'
]].sort_values(['test_number', 'metric']).reset_index(drop=True)

summary_df.to_csv('ab_test_summary_by_test.csv', index=False)
summary_df

,test_number,metric,conversion_rate_1,conversion_rate_2,metric_change_pct,diff_pp,ci_low_pp,ci_high_pp,z_stat,p_value,p_value_adj,significant,significant_adj,result
0,1,add_payment_info / session,0.021251,0.021751,2.352276,0.049989,-0.138957,0.238935,-0.518551,0.604074,0.604074,False,False,Not significant
1,1,add_shipping_info / session,0.041003,0.044078,7.497264,0.307414,0.044517,0.570311,-2.291930,0.021910,0.087639,True,False,Not significant
2,1,begin_checkout / session,0.041048,0.044078,7.381800,0.303005,0.040043,0.565967,-2.258498,0.023915,0.087639,True,False,Not significant
3,1,new_accounts / session,0.084278,0.081451,-3.354299,-0.282692,-0.641788,0.076404,1.542883,0.122859,0.245719,False,False,Not significant
4,2,add_payment_info / session,0.021644,0.022391,3.448865,0.074648,-0.106460,0.255757,-0.807895,0.419151,1.000000,False,False,Not significant
5,2,add_shipping_info / session,0.043328,0.042513,-1.882068,-0.081546,-0.331681,0.168588,0.638943,0.522860,1.000000,False,False,Not significant
6,2,begin_checkout / session,0.043348,0.042532,-1.880854,-0.081531,-0.331720,0.168659,0.638682,0.523030,1.000000,False,False,Not significant
7,2,new_accounts / session,0.082252,0.083274,1.241934,0.102152,-0.237896,0.442199,-0.588793,0.556000,1.000000,False,False,Not significant
8,3,add_payment_info / session,0.025283,0.025384,0.398058,0.010064,-0.154273,0.174401,-0.120029,0.904461,1.000000,False,False,Not significant
9,3,add_shipping_info / session,0.041615,0.041014,-1.443484,-0.060070,-0.268211,0.148070,0.565667,0.571620,1.000000,False,False,Not significant


In [55]:
from google.colab import files
files.download('ab_test_significance_results.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [56]:
files.download('ab_test_summary_by_test.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 10. Висновки

**Перевірка розподілу груп.** У всіх чотирьох тестах сесії розподілились між групами практично порівну (49,9–50,2%), SRM не виявлено, тож результатам можна довіряти.

**Результати в тоталі по тестах** (різниця групи 2 до групи 1 у відсоткових пунктах, 95% довірчий інтервал; значущість після поправки Холма):

- **Тест 1.** Слабкий сигнал покращення на кроках оформлення: `add_shipping_info / session` 4,10% → 4,41% (+0,31 п.п., CI [+0,04; +0,57], p = 0,022) і `begin_checkout / session` 4,10% → 4,41% (+0,30 п.п., CI [+0,04; +0,57], p = 0,024). Після поправки на 4 метрики p_adj = 0,088, тобто ефект **не підтверджується**. `add_payment_info` і `new_accounts` без значущих змін.
- **Тест 2.** Жодних значущих змін (усі p ≥ 0,42).
- **Тест 3.** Жодних значущих змін (усі p ≥ 0,52).
- **Тест 4.** Слабкий сигнал погіршення реєстрацій: `new_accounts / session` 8,55% → 8,26% (−0,29 п.п., CI [−0,52; −0,05], p = 0,018). Після поправки p_adj = 0,070, ефект **не підтверджується**. Інші метрики без змін.

**Особливість даних.** Після підрахунку унікальних сесій `add_shipping_info` і `begin_checkout` майже збігаються: майже кожна сесія, яка почала оформлення, також додала доставку. Тому в тесті 1 це не два незалежні сигнали, а фактично один.

**Розрізи (дослідницький аналіз).** З ~1 800 порівнянь у розрізах country / device / continent / channel значущими без поправки виявились 62 (≈3,5%). Якщо реального ефекту немає, випадково очікується близько 5%. Тож розрізи теж не показують ознак ефекту.

**Загальний висновок.** Жоден із чотирьох тестів не показав статистично надійного впливу на ключові метрики воронки. Розкочувати жоден варіант на підставі цих даних не варто. Тест 1 (можливе покращення оформлення) і тест 4 (можливе погіршення реєстрацій) варто повторити з більшою вибіркою, якщо ці зміни важливі для бізнесу.

**Що змінилось порівняно з першою версією проєкту:**
- метрики рахуються як частка сесій з подією (`COUNT(DISTINCT)`), а не кількість подій. Раніше повторні події в одній сесії завищували конверсію приблизно вдвічі, і найсильніший «результат» тесту 1 (+12,5% до `add_payment_info`, p = 0,00009) виявився артефактом: у групі 2 люди частіше повторно натискали оплату, а не частіше до неї доходили;
- додано перевірку розподілу груп (SRM);
- додано 95% довірчі інтервали для різниці між групами;
- додано поправку Холма на множинні порівняння в межах кожного тесту;
- розрізи явно позначені як дослідницький аналіз.

**Посилання:**
- Дашборд у Tableau:
- CSV з результатами розрахунків:

**Посилання:**
- Дашборд у Tableau: https://public.tableau.com/views/ABTestStatisticalSignificance_17875767365580/Dashboard1?:language=en-GB&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link
- CSV з результатами розрахунків: *https://drive.google.com/file/d/1dfkXWCiKt8MbrvryifMMYwoz8a9GtiTQ/view?usp=sharing*
